# `vacua_vault/` — the local permanent vault

This directory holds your **designated vacuum solutions** — the
`jaxvacua` analogue of a project's output directory for curated
results.  It is created automatically the first time you call
`db.designate_vacua(...)` or `merge_cluster_results(designate=True, ...)`.

Two guarantees:

- `db.clear_cache()` **never** touches this directory (the vault lives
  outside `cache_dir` by design).
- It sits in `.gitignore` at the repo top level, so solutions stay
  private unless you explicitly publish them.

The notebook below inspects whatever is currently stored here.  If the
vault is empty, the cells print an empty inventory — run
`documentation/source/notebooks/05_database_and_infrastructure/25_vacua_storage.ipynb`
first to generate some example designated vacua.

## Directory layout

```
vacua_vault/
├── designated_vacua_catalog.parquet   # index of all designated runs
├── retractions.parquet                # audit trail of retractions
├── archive/                           # opt-in: pre-purge copies
├── KS/
│   └── h12_{N}_model_{M}/             # one dir per local KS model
│       └── shard_0.parquet
├── tdf/
│   └── ks_{X}_tri_{Y}/                # HF-downloaded TDF models
│       └── shard_0.parquet
├── cicy/
│   └── cicy_{X}/                      # HF-downloaded CICY models
│       └── shard_0.parquet
└── custom/                            # fallback for hash-keyed models
    └── {model_hash}/
        └── shard_0.parquet
```

**Note.** Only KS models are loadable locally via `(h12, model_ID)`; CICY geometries are always fetched from the HuggingFace database and therefore land under `cicy/cicy_{X}/`.

## Inspect the current vault

A tiny walk — lists every shard and prints a one-line summary.

In [ ]:
import os
from pathlib import Path

VAULT = Path(__file__).parent if "__file__" in dir() else Path(".").resolve()
# When opened as a notebook, __file__ isn't defined; default to cwd.
if not (VAULT / "designated_vacua_catalog.parquet").exists() and VAULT.name != "vacua_vault":
    VAULT = Path.cwd()

print(f"Vault: {VAULT.resolve()}\n")

shards = sorted(VAULT.rglob("shard_*.parquet"))
if not shards:
    print("(vault is empty — no designated shards yet)")
else:
    try:
        import pandas as pd
    except ImportError:
        print("pandas not available — showing file list only:")
        for s in shards:
            print(f"  {s.relative_to(VAULT)}")
    else:
        print(f"{len(shards)} shard(s):")
        for s in shards:
            try:
                n = len(pd.read_parquet(s))
            except Exception as e:
                n = f"?? ({e.__class__.__name__})"
            print(f"  {s.relative_to(VAULT)}  rows={n}")

## The designated catalog

Top-level summary of every designated entry across all models.

In [ ]:
cat_path = VAULT / "designated_vacua_catalog.parquet"
if not cat_path.exists():
    print("(no catalog — vault is empty or not yet initialised)")
else:
    import pandas as pd
    cat = pd.read_parquet(cat_path)
    print(f"Catalog: {len(cat)} entries\n")
    cols = [c for c in [
        "designated_id", "h12", "ks_id", "triang_id", "cicy_id",
        "model_name", "label", "committed_by", "commit_date",
        "retracted",
    ] if c in cat.columns]
    print(cat[cols].head(20).to_string(index=False))

## Retractions audit trail

Every `retract_designated(...)` call appends a row to `retractions.parquet` — this survives any future catalog rebuild.

In [ ]:
ret_path = VAULT / "retractions.parquet"
if not ret_path.exists():
    print("(no retractions recorded)")
else:
    import pandas as pd
    r = pd.read_parquet(ret_path)
    print(f"{len(r)} retraction(s):")
    print(r.to_string(index=False))

## Next steps

- **Full tutorial:** `documentation/source/notebooks/05_database_and_infrastructure/25_vacua_storage.ipynb` — end-to-end designate → query → retract → purge → upload workflow.
- **Share your vacua:** `db.push_vacua_to_hub(df, label, committed_by, model, create_pr=True)`.  See NB25 §13.
- **Server-side CLI:** `stringjax.vacuavault {validate | rebuild_catalog | curate}` — see NB25 §14.

Any files you see here were produced by `db.designate_vacua(...)` or `merge_cluster_results(..., designate=True, label=...)`.  Safe to delete individual shards manually, but prefer the `purge_retracted(dry_run=False, confirm=True)` CLI to keep the catalog in sync.